In [ ]:
! pip install -q nltk plotly

: 

## Dataset
- **Source:** [Sentiment140 on Kaggle](https://www.kaggle.com/datasets/kazanova/sentiment140)
- **File to upload:** `training.1600000.processed.noemoticon.csv`


In [ ]:
# ── STEP 1 (continued): Import All Libraries ─────────────────────────────────
# An 'import' statement loads an external package into our Python session so
# we can use its functions. Think of it like opening a toolbox before starting work.

import pandas as pd          # pandas: the go-to library for working with tabular data (like spreadsheets)
import numpy as np           # numpy: fast math and array operations
import re                    # re: Python's built-in library for regular expressions (pattern matching in text)
import nltk                  # nltk: Natural Language Toolkit — tools for processing human language
import matplotlib.pyplot as plt  # matplotlib: the foundation plotting library in Python
import seaborn as sns        # seaborn: a prettier, easier-to-use wrapper around matplotlib
import plotly.express as px  # plotly: interactive charts (the user can hover, zoom, etc.)

from google.colab import files                         # lets us upload files from our local computer
from sklearn.model_selection import train_test_split   # splits data into training and testing portions
from sklearn.feature_extraction.text import TfidfVectorizer  # converts text to numbers
from sklearn.linear_model import LogisticRegression , LinearRegression  # the classification model we will train
from sklearn.metrics import classification_report, confusion_matrix  # tools to measure model quality
from nltk.corpus import stopwords                      # common English words to remove ("the", "is", etc.)
from nltk.stem import PorterStemmer                   # reduces words to their root form

# Download the NLTK data file that our cleaning functions need.
# 'stopwords' is a list of very common words that carry little meaning.
nltk.download('stopwords', quiet=True)

# Set a consistent style for all our matplotlib charts
sns.set_theme(style='whitegrid')
print('All libraries imported sucessfully!')



In [ ]:
# ── STEP 2: Upload the Dataset ────────────────────────────────────────────────
# Google Colab runs in the cloud — it doesn't have access to files on our
# computer unless we explicitly upload them. The next line opens a file-picker
# dialog. Click 'Choose Files', find the Sentiment140 CSV, and upload it.
uploaded = files.upload()  # This open the upload diaglog
if not uploaded:
  raise ValueError("No file was uploaded. Please click 'Choose Files', "
        "select the Sentiment140 CSV, and rerun this cell."
  )


  # After uploading, 'uploaded' is a dictionary: { filename: file_bytes }.
# We grab the filename automatically so the code works no matter what we
# named the file on your computer.
filename = list(uploaded.keys())[0]
print(f'Uploaded file: {filename}')


In [ ]:
# ── STEP 2 (continued): Load the Dataset ──────────────────────────────────────
# The Sentiment140 CSV has NO header row — the first row is already data, not
# column names. So we pass header=None and supply the column names ourselves.
# The 6 columns are: sentiment label, tweet id, date, query flag, username, tweet text.
col_names = ['sentiment' , 'id' , 'date' , 'query' , 'user' , 'text']

df = pd.read_csv(filename , encoding = 'latin-1' , header = None , names = col_names)

print(f'Dataset shape: {df.shape}')
print(df.head(3))

In [ ]:
# ── STEP 2 (continued): Sample 50,000 Rows ───────────────────────────────────
# The full dataset has 1.6 million tweets. Training on all of them would take
# many minutes on Colab's free tier. Sampling 50,000 gives us enough data to
# get a good model while keeping training fast (< 2 minutes).
# random_state=42 makes the sample reproducible — you'll get the same rows
# every time you run this cell (42 is just a conventional seed number).

df = df.sample(n=50_000, random_state = 42).reset_index(drop=True)
print(f'Sampled dataset shape: {df.shape}')

In [ ]:
# ── STEP 3: Convert Sentiment Labels ─────────────────────────────────────────
# Sentiment140 uses 0 for negative and 4 for positive.
# Machine learning models prefer labels 0 and 1 (binary), so we remap 4 → 1.
# We also keep only the 'sentiment', 'date', and 'text' columns — the others
# (tweet id, query, username) are not useful for our analysis.

df['sentiment'] = df['sentiment'].map({0: 0, 4:1})
df = df[['sentiment' , 'date' , 'text']]

print('Sentiment value counts:')
print(df['sentiment'].value_counts())  # Should be roughly 25,000 each (balanced dataset)

In [ ]:
# ── STEP 3 (continued): Parse Dates ──────────────────────────────────────────
# The 'date' column is a text string like "Mon Apr 06 22:19:45 PDT 2009".
# We need to convert it to a proper datetime object so we can group tweets by day
# for the trend chart later. errors='coerce' replaces any unparseable dates with NaT
# (Not a Time) instead of crashing.

df['date'] = pd.to_datetime(df['date'], format='%a %b %d %H:%M:%S PDT %Y', errors='coerce')
df['date_only'] = df['date'].dt.date   # Extract just the date portion (no time)
print('Date range:', df['date'].min(), 'to', df['date'].max())

In [ ]:
# ── STEP 4: Define the Text Cleaning Function ─────────────────────────────────
# Raw tweets are very noisy: they contain URLs, @mentions, #hashtags, punctuation,
# and capital letters. None of these help a model learn sentiment — they just add
# noise. We write a function that strips all that away and reduces each word to its
# root form (stemming) so "running", "runs", and "ran" are all treated as "run".

stemmer = PorterStemmer()       # Creates a stemmer objects
stop_words = set(stopwords.words('english'))      # A set of 179 common English words

def clean_tweet(text):
  """Cleans a single tweet string and returns the cleaned version."""
  # 1. Remove URLS(http://... and https://... )
  text = re.sub(r'https?://\S+|www\.\S+', '', text)

  # 2. Remove @mentions (e.g. @eLeonmusk -> '')
  text = re.sub(r'@\w+', '', text)

  # 3. Remove #hashtags (e.g. #AI ->'')
  text = re.sub(r'#\w+', '', text)

  # 4. Remove punctuations and non-alphabetic characters (keep only a-z and spaces)
  text = re.sub(r'[^a-zA-Z\s]', '', text)

  # 5. Convert everything to lowercase so "Chilling" and "chilling" are the same word
  text = text.lower()

  # 6. Split the text into individual words(called 'tokens')
  words = text.split()

  # 7. Remove stopwords and apply stemming
  #    - We skip a word if it's in stop_words (e.g. "the", "and", "is")
  #    - We stem each remaining word to its root (e.g. "loving" → "love")
  words = [stemmer.stem(w) for w in words if w not in stop_words]

  # 8. Join the cleaned words back into a single string
  return ' '.join(words)

print('clean_tweet() function defined')






In [ ]:
# ── STEP 5: Apply Cleaning to the Text Column ────────────────────────────────
# We use pandas .apply() to run our clean_tweet function on every row in the
# 'text' column. This is much faster than writing a Python for-loop.

df['clean_text'] = df['text'].apply(clean_tweet)

#Show a before/after comparision for 3 random tweets
print('=== Before & After Cleanin ===')
for i in range(3):
  print(f'\nOriginal : {df["text"].iloc[i]}')
  print(f'Cleaned : {df["clean_text"].iloc[i]}')
  print('-' * 60)

In [ ]:
# ── STEP 6: TF-IDF Vectorizer Setup ──────────────────────────────────────────
# Machine learning models work with NUMBERS, not words. We need to convert our
# cleaned text into a numerical representation.
#
# TF-IDF stands for Term Frequency – Inverse Document Frequency:
#   • Term Frequency (TF): How often does a word appear in this tweet?
#   • Inverse Document Frequency (IDF): How rare is this word across ALL tweets?
#
# Words that appear frequently in ONE tweet but rarely elsewhere get a HIGH score
# (they are informative). Words that appear in every tweet (like "the") get a LOW
# score (they are not useful for classification).
#
# max_features=50000 means we only keep the 50,000 most informative words.
#
# IMPORTANT — We only *initialise* the vectorizer here.
# We must split the data first and then fit the vectorizer on the training set
# only. Fitting on all data before the split would leak IDF statistics from the
# test set into training, making evaluation results over-optimistic.

tfidf = TfidfVectorizer(max_features = 50_000)
y = df['sentiment'].values  # target label (0 = negative, 1 = positive)

print(f'Labels extracted {len(y)} tweets')


In [ ]:
# ── STEP 7: Train / Test Split and TF-IDF Vectorization ──────────────────────
# We split the RAW TEXT first, then fit TF-IDF only on the training tweets.
# This prevents data leakage: IDF statistics (how rare each word is) should
# be computed from training data only. Using the full dataset before the split
# would let test-set vocabulary influence the model and inflate metrics.
#
# test_size=0.2  → 20% of data goes to the test set, 80% to training
# random_state=42 → reproducible split

X_train_text , X_test_text , y_train, y_test = train_test_split(
    df['clean_text'], y, test_size = 0.2, random_state = 42
)

# fit_transform: Leans vocabulary/IDF from training tweets, then transform them
X_train = tfidf.fit_transform(X_train_text)
# transform: applies the SAME vocabulary/IDF to test tweets (no re-learning)
X_test  = tfidf.transform(X_test_text)


print(f'Training samples : {X_train.shape[0]}')
print(f'Testing samples  : {X_test.shape[0]}')


In [ ]:
# ── STEP 8: Train a Logistic Regression Model ─────────────────────────────────
# We start with Logistic Regression because:
#   1. It's simple and easy to understand — it learns a straight-line decision boundary
#   2. It trains very fast even on large sparse text features
#   3. It performs surprisingly well for text classification tasks
#   4. It gives us probabilities (confidence scores) for each prediction
#
# max_iter=1000 tells the solver to try up to 1000 iterations to find the best
# fit. The default of 100 sometimes isn't enough for text data.
# we can even use different models by changing model(e.g. we can user LinearRegression in place of LogisticRegression)

# model = LinearRegression()

model = LogisticRegression(max_iter = 1000 , random_state = 42)
model.fit(X_train , y_train)  # .fit() is where all the learning happens or model training happens

print('Model Trained!')


In [ ]:
# ── STEP 9: Evaluate the Model ────────────────────────────────────────────────
# Now we test the model on data it has NEVER seen before (the test set).
#
# Classification Report shows:
#   • Precision: Of all tweets predicted as positive, what % were actually positive?
#   • Recall:    Of all actually-positive tweets, what % did the model catch?
#   • F1-score:  Harmonic mean of precision and recall (overall quality per class)
#   • Support:   How many samples of that class exist in the test set

y_pred = model.predict(X_test)

print('===Classification Report ===')
print(classification_report(y_test, y_pred , target_names = ['Negative' , 'Positive']))


In [ ]:
# ── STEP 9 (continued): Confusion Matrix ─────────────────────────────────────
# A confusion matrix is a 2×2 table that shows:
#   • True Negatives  (top-left):  correctly predicted Negative
#   • False Positives (top-right): predicted Positive but was actually Negative
#   • False Negatives (bottom-left): predicted Negative but was actually Positive
#   • True Positives  (bottom-right): correctly predicted Positive
#
# We want the diagonal (correct predictions) to be large and the off-diagonal
# (errors) to be small.

cm = confusion_matrix(y_test , y_pred)

plt.figure(figsize= (6,5))
sns.heatmap(
    cm,
    annot = True,
    fmt = 'd' ,
    cmap = 'Greens' ,
    xticklabels = ['Negative' , 'Positive'] ,
    yticklabels = ['Negative' , 'Positive']
)
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

In [ ]:
# ── STEP 10: Rolling 7-Day Sentiment Trend Chart ──────────────────────────────
# Now for the fun part! We want to see how the average sentiment of tweets
# changes over time. To do this:
#   1. Group all tweets by date and compute the mean sentiment (0–1) for each day
#   2. Apply a 7-day rolling average to smooth out day-to-day noise
#   3. Plot it interactively using Plotly so we can zoom and hover

# Step 10a: Add model's predicted probabilities for positive sentiment
# predict_proba returns a probability for each class; [:, 1] is the positive-class probability
df['sentiment_score'] = model.predict_proba(tfidf.transform(df['clean_text']))[:, 1]

# Step 10b: Group by date and compute daily average sentiment
daily_sentiment = (
    df.groupby('date_only')['sentiment_score'].mean().reset_index().rename(columns = {'sentiment_score': 'avg_sentiment', 'date_only' : 'date'})
)
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])
daily_sentiment = daily_sentiment.sort_values('date')

# Step 10c: Compute 7-day rolling average — each point is the average of that day
# and the 6 days before it. This smooths out spikes and reveals the real trend.
daily_sentiment['rolling_7d'] = daily_sentiment['avg_sentiment'].rolling(window=7 , min_periods=1).mean()

# Step 10d: Plot with Plotly
fig = px.line(
    daily_sentiment,
    x = 'date',
    y = ['avg_sentiment' , 'rolling_7d'],
    labels ={'value': 'Sentiment Score (0 = Negative) , 1 = Positoive' , 'date' : 'Date'},
    title = 'Rolling 7-Day Sentiment Trend Over Time',
    template = 'plotly_white'
)
fig.update_layout(legend_title = 'Metric')
fig.show()
print('\nThe orange line is the smoothed 7-day rolling average - it shows the overall trend.')